# Self-Consistency の効果検証

同じ算数問題に複数の解答を生成し、最終回答の多数決を取る Self-Consistency を試しましょう。サンプリングの1回答と複数回答の多数決で、正答率を比べます。

LLM を動かすために GPU が必要です。


In [1]:
!pip install -q vllm datasets sentencepiece protobuf

## モデルの準備

[Sarashina2.2 1B Instruct](https://huggingface.co/sbintuitions/sarashina2.2-1b-instruct-v0.1) を GPU に読み込みます。Colab では新しい T4 GPU ランタイムで最初から実行してください。


In [2]:
from vllm import LLM, SamplingParams

SEED = 42
model_name = "sbintuitions/sarashina2.2-1b-instruct-v0.1"
llm = LLM(
    model=model_name,
    dtype="half",  # 半精度浮動小数点数 (FP16) を指定
    max_model_len=2048,
    gpu_memory_utilization=0.8,
    seed=SEED,
)

(MainProcess pid=12788) INFO 10-25 12:51:37 [api_utils.py:286] non-default args: {'dtype': 'half', 'seed': 42, 'max_model_len': 2048, 'gpu_memory_utilization': 0.8, 'disable_log_stats': True, 'model': 'sbintuitions/sarashina2.2-1b-instruct-v0.1'}
(MainProcess pid=12788) INFO 10-25 12:51:38 [model.py:733] Resolved architecture: LlamaForCausalLM
(MainProcess pid=12788) WARNING 10-25 12:51:38 [model.py:2498] Casting torch.bfloat16 to torch.float16.
(MainProcess pid=12788) INFO 10-25 12:51:38 [model.py:2152] Using max model len 2048
(MainProcess pid=12788) INFO 10-25 12:51:38 [scheduler.py:307] Chunked prefill is enabled with max_num_batched_tokens=8192.
(MainProcess pid=12788) INFO 10-25 12:51:38 [kernel.py:467] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

(EngineCore pid=13132) INFO 10-25 12:51:43 [core.py:133] Initializing a V1 LLM engine (v0.32.0) with config: model='sbintuitions/sarashina2.2-1b-instruct-v0.1', speculative_config=None, tokenizer='sbintuitions/sarashina2.2-1b-instruct-v0.1', skip_tokenizer_init=False, tokenizer_mode=auto, revision=main, tokenizer_revision=main, trust_remote_code=False, dtype=torch.float16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=False, aux_output_config=AuxOutputConfig(enable_return_routed_experts=False, max_bytes=None), kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), obse

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


(EngineCore pid=13132) WARNING 10-25 12:51:55 [utils.py:333] Attempted to load weight lm_head.weight with dtype torch.bfloat16 into parameter with dtype torch.float16
(EngineCore pid=13132) WARNING 10-25 12:51:57 [utils.py:333] Attempted to load weight embed_tokens.weight with dtype torch.bfloat16 into parameter with dtype torch.float16
(EngineCore pid=13132) WARNING 10-25 12:52:00 [utils.py:333] Attempted to load weight layers.0.input_layernorm.weight with dtype torch.bfloat16 into parameter with dtype torch.float16
(EngineCore pid=13132) WARNING 10-25 12:52:00 [utils.py:333] Attempted to load weight layers.0.mlp.down_proj.weight with dtype torch.bfloat16 into parameter with dtype torch.float16
(EngineCore pid=13132) WARNING 10-25 12:52:00 [utils.py:333] Attempted to load weight layers.0.post_attention_layernorm.weight with dtype torch.bfloat16 into parameter with dtype torch.float16
(EngineCore pid=13132) WARNING 10-25 12:52:00 [utils.py:333] Attempted to load weight layers.0.self_at

Capturing CUDA graphs (FULL): 100%|██████████| 2/2 [00:00<00:00, 16.28it/s]


(EngineCore pid=13132) INFO 10-25 12:52:20 [model_runner.py:1127] Graph capturing finished in 5 secs, took 0.25 GiB
(EngineCore pid=13132) INFO 10-25 12:52:21 [gpu_worker.py:759] Available KV cache memory: 8.11 GiB
(EngineCore pid=13132) INFO 10-25 12:52:21 [gpu_worker.py:774] CUDA graph memory profiling is enabled (default since v0.21.0). The current --gpu-memory-utilization=0.8000 is equivalent to --gpu-memory-utilization=0.7800 without CUDA graph memory profiling. To maintain the same effective KV cache size as before, increase --gpu-memory-utilization to 0.8200. To disable, set VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=0.
(EngineCore pid=13132) INFO 10-25 12:52:21 [kv_cache_utils.py:2327] GPU KV cache size: 101,200 tokens, Maximum concurrency for 2,048 tokens per request: 49.41x
(EngineCore pid=13132) INFO 10-25 12:52:21 [kernel_warmup.py:176] JIT kernel warmup starting.
(EngineCore pid=13132) INFO 10-25 12:52:21 [kernel_warmup.py:189] JIT kernel warmup finished in 0.00s.


Capturing CUDA graphs (FULL): 100%|██████████| 35/35 [00:01<00:00, 20.65it/s]


(EngineCore pid=13132) INFO 10-25 12:52:28 [model_runner.py:1127] Graph capturing finished in 7 secs, took 0.13 GiB
(EngineCore pid=13132) INFO 10-25 12:52:28 [gpu_worker.py:971] CUDA graph pool memory: 0.13 GiB (actual), 0.29 GiB (estimated), difference: 0.16 GiB (125.8%).
(EngineCore pid=13132) INFO 10-25 12:52:28 [gpu_worker.py:1034] Free memory on device (14.46/14.56 GiB) on startup. Desired GPU memory utilization is (0.8, 11.65 GiB). Actual usage is 2.88 GiB for consumed memory (weights + non-torch), 0.66 GiB for peak activation, and 0.13 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=8409731175` (7.83 GiB) to fit into requested memory, or `--kv-cache-memory=11427179008` (10.64 GiB) to fully utilize gpu memory. Current kv cache memory in use is 8.11 GiB.
(EngineCore pid=13132) INFO 10-25 12:52:30 [jit_monitor.py:84] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
(EngineCore pid=13132) INFO 10-2

## 問題と最終回答

[日本語の算数問題](https://huggingface.co/datasets/ryo0634/gsm8k-ja-filtered-dev)の先頭100問を使います。正解の数値は生成後の採点にだけ使います。


In [3]:
from datasets import load_dataset

NUM_PROBLEMS = 100
dataset = load_dataset("ryo0634/gsm8k-ja-filtered-dev", split=f"dev[:{NUM_PROBLEMS}]")
print("問題数:", len(dataset))
print("問題:", dataset[0]["question"])
print("正解:", dataset[0]["answer_number"])


問題数: 100
問題: ビルは3階からスタートします。彼はエレベーターで、スタートした階の4倍に6を足した階まで上がります。ビルは今、何階にいますか？
正解: 18


モデルの出力文字列の最後に出現した数値を、問題の回答として抽出します。`Fraction` で `1,000` と `1000.0`、`1/2` と `0.5` を同じ値に揃えます。読み取れなければ `None` にします。


In [4]:
import re
from fractions import Fraction


def extract_answer(text: str) -> str | None:
    """最後に出現する数値の抽出と分数表現への統一"""
    matches = re.findall(r"[-+]?\d[\d,]*(?:\.\d+)?(?:\s*/\s*[-+]?\d+)?", text)
    if not matches:
        return None
    try:
        parts = matches[-1].replace(",", "").split("/")
        value = Fraction(parts[0].strip())  # 小数も正確な分数として扱う
        if len(parts) == 2:
            value /= Fraction(parts[1].strip())
        return str(value)
    except (ValueError, ZeroDivisionError):
        return None


for text in ["答え：1,000", "答えは 1000.0", "100 ÷ 200 = 1/2", "0.500"]:
    print(repr(text), "→", extract_answer(text))


'答え：1,000' → 1000
'答えは 1000.0' → 1000
'100 ÷ 200 = 1/2' → 1/2
'0.500' → 1/2


## GPU で回答を生成する

vLLM に質問をまとめて渡し、各問5回答をサンプリングします。プロンプトには計算の過程と最終回答を書くよう指示します。
生成関数からは、質問ごとの回答文字列のリストを返します。


In [5]:
def generate_answers(questions: list[str], count: int) -> list[list[str]]:
    """質問ごとの複数の回答文字列の生成"""
    instruction = (
        "次の算数問題を、計算の過程を示して順に解いてください。"
        "最後に「答え：数値」と書いてください。\n\n"
    )
    messages = [
        [{"role": "user", "content": instruction + question}]
        for question in questions
    ]
    params = SamplingParams(
        n=count,
        temperature=1.0,
        max_tokens=512,
        seed=SEED,
    )
    outputs = llm.chat(messages, sampling_params=params)
    return [[answer.text for answer in output.outputs] for output in outputs]


## 1回答の正答率

後で多数決にも使うため、先に各問5回答を生成し、その1つ目で採点します。


In [6]:
texts = generate_answers(list(dataset["question"]), count=5) # count で回答生成数を指定
model_answers = [[extract_answer(text) for text in group] for group in texts]
gold_answers = [extract_answer(str(number)) for number in dataset["answer_number"]]

Rendering conversations:   0%|          | 0/100 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 500/500 [01:09<00:00,  7.19it/s, est. speed input: 585.12 toks/s, output: 1499.15 toks/s]


In [7]:
# １つめの回答を使って採点
single_accuracy = sum(
    answers[0] == gold for answers, gold in zip(model_answers, gold_answers)
) / len(dataset)
print(f"1回答の正答率: {single_accuracy:.1%}")

1回答の正答率: 71.0%


## 多数決

読み取れた答えのうち最多票のものを選びます。過半数は必須にしません。同票や全件読み取り失敗なら棄権し、不正解に数えます。


In [8]:
from collections import Counter


def majority_vote(answers: list[str | None]) -> str | None:
    """有効な回答の最多票と同票時の棄権"""
    ranking = Counter(answer for answer in answers if answer is not None).most_common()
    if not ranking or (len(ranking) > 1 and ranking[0][1] == ranking[1][1]):
        return None
    return ranking[0][0]


majority_answers = [majority_vote(answers) for answers in model_answers]
majority_accuracy = sum(
    answer == gold for answer, gold in zip(majority_answers, gold_answers)
) / len(dataset)
print(f"多数決の正答率: {majority_accuracy:.1%}")

多数決の正答率: 84.0%


1回答と多数決の正答率を見比べ、今回の実験で改善したかを確認しましょう。

## まとめと次の実験

Self-Consistency は複数の解答の最終回答を集約します。同じ誤答が集まることもあり、精度が上がる保証はありません。
`texts[0]` と `model_answers[0]` を表示すると、最初の問題の生成文と、抽出した数値を確認できます。

- 回答生成数 (`count`) を変化させると、正答率はどう変わるでしょうか。
- デコーディング時のパラメータ（`temperature` など）を変えたときも、同じ傾向になるでしょうか。

参考：[Self-Consistency の論文](https://arxiv.org/abs/2203.11171)
